In [ ]:


import os
import glob
import json
import shutil
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, Model

print("TensorFlow:", tf.__version__)

# ========================== CONFIG ==========================
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
SSL_EPOCHS = 30
TEMPERATURE = 0.5
AUTOTUNE = tf.data.AUTOTUNE
COMBINED_PATH = "/kaggle/input/datasets/sabbir4724/rose-leaf"

WORK = "/kaggle/working"
ENCODER_PATH = f"{WORK}/ssl_encoder.keras"
CKPT_WEIGHTS = f"{WORK}/simclr_ckpt.weights.h5"
CKPT_STATE = f"{WORK}/simclr_ckpt_state.json"

# ========================== SKIP IF ENCODER ALREADY EXISTS ==========================
if os.path.exists(ENCODER_PATH):
    print(f"Found {ENCODER_PATH} — SimCLR pretraining skip.")
else:
    found = glob.glob("/kaggle/input/**/ssl_encoder.keras", recursive=True)
    if found:
        shutil.copy(found[0], ENCODER_PATH)
        print(f"Copied encoder from {found[0]} -> {ENCODER_PATH} — pretraining skip.")

if not os.path.exists(ENCODER_PATH):
    # ========================== FILE LIST + SAME SPLIT AS EVERYWHERE ==========================
    class_names = sorted(os.listdir(COMBINED_PATH))
    class_to_idx = {c: i for i, c in enumerate(class_names)}
    image_paths, labels = [], []
    for cls in class_names:
        folder = os.path.join(COMBINED_PATH, cls)
        for img in os.listdir(folder):
            image_paths.append(os.path.join(folder, img))
            labels.append(class_to_idx[cls])

    train_paths, _, train_labels, _ = train_test_split(
        image_paths, labels, test_size=0.2, random_state=SEED, stratify=labels
    )
    print("SimCLR pretraining images (labels NOT used):", len(train_paths))

    def load_image(path):
        img = tf.io.read_file(path)
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
        img = tf.cast(img, tf.float32)
        img.set_shape([IMG_SIZE, IMG_SIZE, 3])
        return img

    train_ds = (tf.data.Dataset.from_tensor_slices(train_paths)
                .map(load_image, num_parallel_calls=AUTOTUNE)
                .shuffle(2000)
                .batch(BATCH_SIZE)
                .prefetch(AUTOTUNE))

    # ========================== AUGMENTATION ==========================
    augmenter = tf.keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.2),
        layers.RandomZoom(0.2),
        layers.RandomContrast(0.2),
        layers.RandomTranslation(0.1, 0.1),
    ])

    # ========================== SIMCLR MODEL ==========================
    base_encoder = tf.keras.applications.MobileNetV2(
        include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base_encoder.trainable = True

    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
    x = base_encoder(x, training=True)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation="relu")(x)
    projection = layers.Dense(128)(x)
    simclr_model = Model(inputs, projection)

    # ========================== NT-XENT LOSS ==========================
    def nt_xent_loss(zis, zjs, temperature=0.5):
        zis = tf.math.l2_normalize(zis, axis=1)
        zjs = tf.math.l2_normalize(zjs, axis=1)
        representations = tf.concat([zis, zjs], axis=0)
        similarity_matrix = tf.matmul(representations, representations, transpose_b=True)
        batch_size = tf.shape(zis)[0]
        mask = tf.eye(2 * batch_size)
        similarity_matrix = similarity_matrix - mask * 1e9
        positives = tf.concat([tf.range(batch_size, 2 * batch_size), tf.range(0, batch_size)], axis=0)
        logits = similarity_matrix / temperature
        loss = tf.keras.losses.sparse_categorical_crossentropy(positives, logits, from_logits=True)
        return tf.reduce_mean(loss)

    optimizer = tf.keras.optimizers.Adam(1e-4)

    @tf.function
    def train_step(images):
        aug1 = augmenter(images, training=True)
        aug2 = augmenter(images, training=True)
        with tf.GradientTape() as tape:
            z1 = simclr_model(aug1, training=True)
            z2 = simclr_model(aug2, training=True)
            loss = nt_xent_loss(z1, z2, TEMPERATURE)
        grads = tape.gradient(loss, simclr_model.trainable_variables)
        optimizer.apply_gradients(zip(grads, simclr_model.trainable_variables))
        return loss

    # ========================== RESUME FROM CHECKPOINT ==========================
    ssl_losses, start_epoch = [], 0
    if os.path.exists(CKPT_STATE) and os.path.exists(CKPT_WEIGHTS):
        with open(CKPT_STATE) as f:
            state = json.load(f)
        ssl_losses, start_epoch = state["losses"], state["epochs_done"]
        simclr_model.load_weights(CKPT_WEIGHTS)
        print(f"Resuming SimCLR from epoch {start_epoch}/{SSL_EPOCHS} "
              f"(optimizer state reset hoy, weights restore hoy)")

    # ========================== TRAINING LOOP ==========================
    for epoch in range(start_epoch, SSL_EPOCHS):
        batch_losses = []
        for images in train_ds:
            batch_losses.append(train_step(images).numpy())
        epoch_loss = float(np.mean(batch_losses))
        ssl_losses.append(epoch_loss)

        simclr_model.save_weights(CKPT_WEIGHTS)
        with open(CKPT_STATE, "w") as f:
            json.dump({"epochs_done": epoch + 1, "losses": ssl_losses}, f)
        print(f"Epoch {epoch+1}/{SSL_EPOCHS}  Loss={epoch_loss:.4f}  (checkpoint saved)")

    # ========================== SAVE ENCODER + LOSS PLOT ==========================
    base_encoder.save(ENCODER_PATH)
    print("Encoder Saved:", ENCODER_PATH)

    plt.figure(figsize=(8, 5))
    plt.plot(range(1, len(ssl_losses) + 1), ssl_losses)
    plt.title("SimCLR NT-Xent Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.savefig(f"{WORK}/simclr_loss.png", dpi=600, bbox_inches="tight")
    plt.show()

print("\nReady:", ENCODER_PATH, "| exists =", os.path.exists(ENCODER_PATH))

In [ ]:
# ==================================================================================
# RoseSSL-CBAMNet (Proposed) — LABEL EFFICIENCY ANALYSIS
# ----------------------------------------------------------------------------------
#==================================================================================

import os
import gc
import math
import time
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
from tensorflow.keras import layers, Model

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import label_binarize

print("TensorFlow:", tf.__version__)

# ========================== CONFIG ==========================
IMG_SIZE = 224
BATCH_SIZE = 32
COMBINED_PATH = "/kaggle/input/datasets/sabbir4724/rose-leaf"
SSL_ENCODER_PATH = "/kaggle/working/ssl_encoder.keras"
OUTPUT_DIR = "/kaggle/working/label_efficiency_proposed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FIXED_TEST_SEED = 42
VAL_FRAC = 0.10
LABEL_FRACTIONS = [0.01, 0.05, 0.10, 0.25, 0.50, 1.00]
SEEDS = [42, 123, 2024]                           # final paper run e 5 seed dao: [42, 123, 2024, 7, 1337]
MIN_PER_CLASS = 2

CLASSIFIER_EPOCHS = 30
MIN_TOTAL_STEPS = 300                             
MAX_EPOCHS = 150
FREEZE_BACKBONE = False                          

RESULTS_CSV = os.path.join(OUTPUT_DIR, "label_efficiency_runs.csv")
AUTOTUNE = tf.data.AUTOTUNE

# ========================== FILE LIST ==========================
class_names = sorted(os.listdir(COMBINED_PATH))
class_to_idx = {c: i for i, c in enumerate(class_names)}
NUM_CLASSES = len(class_names)

image_paths, labels = [], []
for cls in class_names:
    folder = os.path.join(COMBINED_PATH, cls)
    for img in os.listdir(folder):
        image_paths.append(os.path.join(folder, img))
        labels.append(class_to_idx[cls])
image_paths = np.array(image_paths)
labels = np.array(labels)
print("Classes:", NUM_CLASSES, class_names)
print("Total images:", len(image_paths))

# ========================== FIXED SPLITS ==========================
trainpool_paths, test_paths, trainpool_labels, test_labels = train_test_split(
    image_paths, labels, test_size=0.20, random_state=FIXED_TEST_SEED, stratify=labels
)
pool_paths, val_paths, pool_labels, val_labels = train_test_split(
    trainpool_paths, trainpool_labels, test_size=VAL_FRAC,
    random_state=FIXED_TEST_SEED, stratify=trainpool_labels
)
print(f"Label pool: {len(pool_paths)} | Fixed val: {len(val_paths)} | Fixed test: {len(test_paths)}")
print("Labeled subset sizes:",
      {f"{int(f*100)}%": max(int(round(f * len(pool_paths))), MIN_PER_CLASS * NUM_CLASSES)
       for f in LABEL_FRACTIONS})

# ========================== DATA PIPELINE ==========================
def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32)
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    return img, label

def make_dataset(paths, labs, shuffle=False, seed=None):
    ds = tf.data.Dataset.from_tensor_slices((paths, labs))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(2000, seed=seed)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

val_ds = make_dataset(val_paths, val_labels)
test_ds = make_dataset(test_paths, test_labels)

# ========================== CBAM ==========================
class ChannelMean(layers.Layer):
    def call(self, x):
        return tf.reduce_mean(x, axis=-1, keepdims=True)

class ChannelMax(layers.Layer):
    def call(self, x):
        return tf.reduce_max(x, axis=-1, keepdims=True)

def cbam_block(input_feature, ratio=8):
    channel = input_feature.shape[-1]
    avg_pool = layers.Reshape((1, 1, channel))(layers.GlobalAveragePooling2D()(input_feature))
    max_pool = layers.Reshape((1, 1, channel))(layers.GlobalMaxPooling2D()(input_feature))
    d1 = layers.Dense(channel // ratio, activation="relu")
    d2 = layers.Dense(channel)
    ch_att = layers.Activation("sigmoid")(layers.Add()([d2(d1(avg_pool)), d2(d1(max_pool))]))
    ch_refined = layers.Multiply()([input_feature, ch_att])
    concat = layers.Concatenate(axis=-1)([ChannelMean()(ch_refined), ChannelMax()(ch_refined)])
    sp_att = layers.Conv2D(1, 7, padding="same", activation="sigmoid")(concat)
    return layers.Multiply()([ch_refined, sp_att])

# ========================== PROPOSED MODEL (SimCLR + MobileNetV2 + CBAM) ==========================
def build_proposed(seed):
    tf.keras.utils.set_random_seed(seed)
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
    backbone = tf.keras.models.load_model(SSL_ENCODER_PATH)
    backbone.trainable = not FREEZE_BACKBONE
    x = backbone(x, training=False)
    x = cbam_block(x, ratio=8)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    model = Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

# ========================== STRATIFIED LABEL SUBSET ==========================
def sample_label_subset(fraction, seed):
    n_total = len(pool_paths)
    if fraction >= 1.0:
        return np.arange(n_total)
    n = max(int(round(fraction * n_total)), MIN_PER_CLASS * NUM_CLASSES)
    n = min(n, n_total - NUM_CLASSES)
    idx, _ = train_test_split(np.arange(n_total), train_size=n, stratify=pool_labels, random_state=seed)
    return idx

# ========================== ONE RUN ==========================
def run_one(fraction, seed):
    idx = sample_label_subset(fraction, seed)
    train_ds = make_dataset(pool_paths[idx], pool_labels[idx], shuffle=True, seed=seed)
    steps_per_epoch = math.ceil(len(idx) / BATCH_SIZE)
    epochs = int(min(MAX_EPOCHS, max(CLASSIFIER_EPOCHS, math.ceil(MIN_TOTAL_STEPS / steps_per_epoch))))

    model = build_proposed(seed)
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3, verbose=0),
    ]
    t0 = time.time()
    hist = model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=callbacks, verbose=0)
    train_time = time.time() - t0

    y_prob = model.predict(test_ds, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    y_true = test_labels

    y_bin = label_binarize(y_true, classes=np.arange(NUM_CLASSES))
    try:
        auc_m = roc_auc_score(y_bin, y_prob, average="macro", multi_class="ovr")
    except ValueError:
        auc_m = np.nan

    row = {
        "fraction": fraction, "seed": seed, "n_labeled": len(idx),
        "epochs_run": len(hist.history["loss"]),
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_auc": auc_m,
        "train_time_s": round(train_time, 1),
    }
    per_class = f1_score(y_true, y_pred, average=None, labels=np.arange(NUM_CLASSES), zero_division=0)
    for c, v in zip(class_names, per_class):
        row[f"f1_{c}"] = v

    del model
    tf.keras.backend.clear_session()
    gc.collect()
    return row

# ========================== RESUMABLE MAIN LOOP ==========================
if os.path.exists(RESULTS_CSV):
    runs_df = pd.read_csv(RESULTS_CSV)
    print(f"Resuming... {len(runs_df)} runs already finished")
else:
    runs_df = pd.DataFrame()

def is_done(frac, seed):
    if runs_df.empty:
        return False
    return bool(((np.isclose(runs_df["fraction"], frac)) & (runs_df["seed"] == seed)).any())

total_runs = len(SEEDS) * len(LABEL_FRACTIONS)
for seed in SEEDS:
    for frac in LABEL_FRACTIONS:
        if is_done(frac, seed):
            continue
        row = run_one(frac, seed)
        runs_df = pd.concat([runs_df, pd.DataFrame([row])], ignore_index=True)
        runs_df.to_csv(RESULTS_CSV, index=False)
        print(f"[{len(runs_df)}/{total_runs}] frac={frac:>5.0%} seed={seed:<5d} n={row['n_labeled']:<5d} "
              f"ep={row['epochs_run']:<3d} Acc={row['accuracy']:.4f} F1(w)={row['f1_weighted']:.4f} "
              f"AUC={row['macro_auc']:.4f}")

runs_df = pd.read_csv(RESULTS_CSV)

# ==================================================================================
# ANALYSIS
# ==================================================================================
print("\n" + "=" * 78)
print("LABEL EFFICIENCY — RoseSSL-CBAMNet (mean ± std over seeds, fixed held-out test set)")
print("=" * 78)

metrics = ["accuracy", "f1_weighted", "f1_macro", "macro_auc"]
g = runs_df.groupby("fraction")
agg = g[metrics].agg(["mean", "std"])
agg.columns = [f"{m}_{s}" for m, s in agg.columns]
agg["n_labeled"] = g["n_labeled"].first()
agg["n_seeds"] = g["seed"].nunique()
agg = agg.reset_index().sort_values("fraction")
agg.to_csv(os.path.join(OUTPUT_DIR, "label_efficiency_summary.csv"), index=False)

# ---- paper-style table ----
tbl = pd.DataFrame({"Labels used": [f"{f:.0%}" for f in agg["fraction"]], "n_labeled": agg["n_labeled"].values})
for m, name in [("accuracy", "Accuracy (%)"), ("f1_weighted", "Weighted F1 (%)"),
                ("f1_macro", "Macro F1 (%)"), ("macro_auc", "Macro AUC (%)")]:
    tbl[name] = [f"{100*a:.2f} ± {100*np.nan_to_num(b):.2f}"
                 for a, b in zip(agg[f"{m}_mean"], agg[f"{m}_std"])]
tbl.to_csv(os.path.join(OUTPUT_DIR, "table_label_efficiency.csv"), index=False)
print(tbl.to_string(index=False))

# ---- % of full-label performance retained + marginal gain ----
full_acc = agg[np.isclose(agg["fraction"], 1.0)]["accuracy_mean"].values[0]
full_f1 = agg[np.isclose(agg["fraction"], 1.0)]["f1_weighted_mean"].values[0]
agg["acc_retained_pct"] = 100 * agg["accuracy_mean"] / full_acc
agg["f1_retained_pct"] = 100 * agg["f1_weighted_mean"] / full_f1
agg["marginal_acc_gain_pp"] = agg["accuracy_mean"].diff() * 100
agg[["fraction", "n_labeled", "acc_retained_pct", "f1_retained_pct", "marginal_acc_gain_pp"]].to_csv(
    os.path.join(OUTPUT_DIR, "retention_and_marginal_gain.csv"), index=False)
print("\nPerformance retained vs full-label model, and marginal gain per extra label step:")
print(agg[["fraction", "n_labeled", "acc_retained_pct", "f1_retained_pct", "marginal_acc_gain_pp"]]
      .round(3).to_string(index=False))

# ---- smallest label fraction reaching X% of full-label accuracy ----
need_rows = []
for pct in [90, 95, 98, 99]:
    ok = agg[agg["accuracy_mean"] >= (pct / 100.0) * full_acc]
    f = float(ok["fraction"].iloc[0]) if len(ok) else np.nan
    need_rows.append({
        "target": f"{pct}% of full-label accuracy",
        "min_label_fraction": f,
        "n_labeled": int(ok["n_labeled"].iloc[0]) if len(ok) else np.nan,
        "label_reduction_x": (1.0 / f) if (f == f and f > 0) else np.nan,
    })
need_df = pd.DataFrame(need_rows)
need_df.to_csv(os.path.join(OUTPUT_DIR, "labels_needed_for_target.csv"), index=False)
print(f"\nLabels needed (full-label accuracy = {full_acc:.4f}):")
print(need_df.round(4).to_string(index=False))

# ---- normalized label-efficiency AUC (area under accuracy vs log10 fraction) ----
_trapz = getattr(np, "trapezoid", None) or np.trapz
lx = np.log10(agg["fraction"].values)
le_auc = _trapz(agg["accuracy_mean"].values, lx) / (lx.max() - lx.min())
print(f"\nNormalized label-efficiency AUC (accuracy): {le_auc:.4f}")
pd.DataFrame([{"LE_AUC_normalized_accuracy": le_auc}]).to_csv(
    os.path.join(OUTPUT_DIR, "le_auc.csv"), index=False)

# ---- per-class F1 vs fraction ----
pc_cols = [f"f1_{c}" for c in class_names]
pc = runs_df.groupby("fraction")[pc_cols].mean().sort_index()
pc.columns = class_names
pc.to_csv(os.path.join(OUTPUT_DIR, "per_class_f1_vs_fraction.csv"))
print("\nPer-class F1 (mean over seeds):")
print(pc.round(4).to_string())

# ==================================================================================
# PLOTS (600 DPI)
# ==================================================================================
xticks = [f * 100 for f in agg["fraction"]]

def style_xaxis(ax):
    ax.set_xscale("log")
    ax.xaxis.set_major_formatter(ScalarFormatter())
    ax.set_xticks(xticks)
    ax.set_xticklabels([f"{x:g}%" for x in xticks])
    ax.minorticks_off()
    ax.grid(alpha=0.25)
    ax.set_xlabel("Labeled training data (% of label pool)")

# Fig 1: label efficiency curves
fig, ax = plt.subplots(figsize=(8.5, 5.5))
x = agg["fraction"].values * 100
for m, lab, col, mk in [("accuracy", "Accuracy", "#C44E52", "o"),
                        ("f1_weighted", "Weighted F1", "#4C72B0", "^"),
                        ("macro_auc", "Macro AUC", "#55A868", "s")]:
    mu = agg[f"{m}_mean"].values
    sd = np.nan_to_num(agg[f"{m}_std"].values)
    ax.plot(x, mu, marker=mk, lw=2, ms=7, color=col, label=lab)
    ax.fill_between(x, mu - sd, mu + sd, color=col, alpha=0.15)
ax.axhline(full_acc, color="gray", ls=":", lw=1.2, label="Full-label accuracy")
style_xaxis(ax)
ax.set_ylabel("Test score")
ax.set_title("Label Efficiency — RoseSSL-CBAMNet")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "label_efficiency_curve.png"), dpi=600, bbox_inches="tight")
plt.show()

# Fig 2: % of full-label performance retained
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar([f"{f:.0%}" for f in agg["fraction"]], agg["acc_retained_pct"], color="#C44E52", alpha=0.85)
for b, v in zip(bars, agg["acc_retained_pct"]):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.3, f"{v:.1f}%", ha="center", fontsize=9)
ax.set_ylim(max(0, agg["acc_retained_pct"].min() - 10), 103)
ax.set_xlabel("Labeled training data")
ax.set_ylabel("Accuracy retained vs full-label model (%)")
ax.set_title("Performance Retained with Fewer Labels")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "performance_retained.png"), dpi=600, bbox_inches="tight")
plt.show()

# Fig 3: per-class F1 heatmap
fig, ax = plt.subplots(figsize=(1.6 * len(pc) + 3, 0.9 * NUM_CLASSES + 2))
im = ax.imshow(pc.T.values, aspect="auto", cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(len(pc)))
ax.set_xticklabels([f"{f:.0%}" for f in pc.index])
ax.set_yticks(range(NUM_CLASSES))
ax.set_yticklabels(class_names)
for i in range(NUM_CLASSES):
    for j in range(len(pc)):
        v = pc.T.values[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=9,
                color="white" if v < 0.6 else "black")
ax.set_xlabel("Labeled training data")
ax.set_title("Per-class F1 vs Label Fraction")
plt.colorbar(im, ax=ax, label="F1")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "per_class_f1_heatmap.png"), dpi=600, bbox_inches="tight")
plt.show()

print("\n" + "=" * 78)
print(f"All outputs saved under: {OUTPUT_DIR}/")
print("=" * 78)
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(" -", f)

In [ ]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

OUTPUT_DIR = "/kaggle/working/label_efficiency_proposed"
CSV_PATH = os.path.join(OUTPUT_DIR, "per_class_f1_vs_fraction.csv")

CMAP = "RdYlGn"        # <-- colour ekhane bodlao
VMIN, VMAX = 0.60, 1.00  # <-- range chhoto korle difference beshi spotlight hoy

pc = pd.read_csv(CSV_PATH, index_col=0)
class_names = list(pc.columns)
fractions = list(pc.index)
M = pc.T.values  # rows = classes, cols = fractions

cmap = plt.get_cmap(CMAP)
norm = Normalize(vmin=VMIN, vmax=VMAX)

fig, ax = plt.subplots(figsize=(1.6 * len(fractions) + 3, 0.9 * len(class_names) + 2))
im = ax.imshow(M, aspect="auto", cmap=cmap, norm=norm)

ax.set_xticks(range(len(fractions)))
ax.set_xticklabels([f"{float(f):.0%}" for f in fractions], fontsize=11)
ax.set_yticks(range(len(class_names)))
ax.set_yticklabels(class_names, fontsize=11)

# cell text: background brightness onujayi black/white
for i in range(len(class_names)):
    for j in range(len(fractions)):
        v = M[i, j]
        r, g, b, _ = cmap(norm(v))
        lum = 0.299 * r + 0.587 * g + 0.114 * b
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=11,
                fontweight="bold", color="white" if lum < 0.45 else "black")

# white grid lines between cells
ax.set_xticks(np.arange(-0.5, len(fractions), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(class_names), 1), minor=True)
ax.grid(which="minor", color="white", linewidth=2)
ax.tick_params(which="minor", length=0)

ax.set_xlabel("Labeled training data", fontsize=12)
ax.set_title("Per-class F1 vs Label Fraction — RoseSSL-CBAMNet", fontsize=13, fontweight="bold")
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("F1-score", fontsize=11)

plt.tight_layout()
out = os.path.join(OUTPUT_DIR, "per_class_f1_heatmap_recolor.png")
plt.savefig(out, dpi=600, bbox_inches="tight")
plt.show()
print("Saved:", out)

In [ ]:
# ==================================================================================
# ALL-SEED COMPARISON IN ONE FIGURE + BEST-SEED RANKING
# Reads /kaggle/working/label_efficiency_proposed/label_efficiency_runs.csv
# Output: label_efficiency_seed_comparison.png (600 DPI) + seed_ranking.csv
# ==================================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

OUTPUT_DIR = "/kaggle/working/label_efficiency_proposed"
runs = pd.read_csv(os.path.join(OUTPUT_DIR, "label_efficiency_runs.csv"))

seeds = sorted(runs["seed"].unique())
fractions = sorted(runs["fraction"].unique())
x = np.array(fractions) * 100
METRICS = [("accuracy", "Accuracy"), ("f1_weighted", "Weighted F1"), ("macro_auc", "Macro AUC")]

# ---------- seed ranking: rank per (fraction, metric), then average ----------
rank_parts = []
for metric, _ in METRICS:
    piv = runs.pivot_table(index="fraction", columns="seed", values=metric)
    rank_parts.append(piv.rank(axis=1, ascending=False, method="average"))
mean_rank = pd.concat(rank_parts).mean(axis=0)

summary = pd.DataFrame({
    "seed": seeds,
    "mean_accuracy_pct": [100 * runs[runs.seed == s]["accuracy"].mean() for s in seeds],
    "mean_f1_weighted_pct": [100 * runs[runs.seed == s]["f1_weighted"].mean() for s in seeds],
    "mean_macro_auc_pct": [100 * runs[runs.seed == s]["macro_auc"].mean() for s in seeds],
    "avg_rank_all_fractions_metrics": [mean_rank[s] for s in seeds],
}).sort_values("avg_rank_all_fractions_metrics").reset_index(drop=True)
summary.to_csv(os.path.join(OUTPUT_DIR, "seed_ranking.csv"), index=False)
best_seed = int(summary.loc[0, "seed"])

print("SEED RANKING (lower avg rank = better):")
print(summary.round(3).to_string(index=False))
print(f"\nBest seed overall: {best_seed}")

# ---------- figure ----------
colors = {s: c for s, c in zip(seeds, ["#4C72B0", "#55A868", "#DD8452", "#8172B3", "#937860", "#DA8BC3"])}
markers = ["o", "s", "^", "D", "v", "P"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (metric, title) in zip(axes.flat[:3], METRICS):
    for i, s in enumerate(seeds):
        sub = runs[runs.seed == s].sort_values("fraction")
        lab = f"Seed {s}" + (" (best)" if s == best_seed else "")
        ax.plot(sub["fraction"] * 100, sub[metric], marker=markers[i % len(markers)], lw=1.8,
                ms=6, alpha=0.85, color=colors[s], label=lab)
    g = runs.groupby("fraction")[metric]
    mu, sd = g.mean().values, np.nan_to_num(g.std().values)
    ax.plot(x, mu, color="black", lw=2.6, ls="--", label="Mean")
    ax.fill_between(x, mu - sd, mu + sd, color="black", alpha=0.10)
    ax.set_xscale("log")
    ax.xaxis.set_major_formatter(ScalarFormatter())
    ax.set_xticks(x)
    ax.set_xticklabels([f"{v:g}%" for v in x])
    ax.minorticks_off()
    ax.grid(alpha=0.25)
    ax.set_xlabel("Labeled training data")
    ax.set_ylabel(title)
    ax.set_title(f"{title} vs Label Fraction (per seed)")
    ax.legend(fontsize=8, loc="lower right")

# heatmap: accuracy (seed x fraction)
ax = axes.flat[3]
H = runs.pivot_table(index="seed", columns="fraction", values="accuracy").reindex(index=seeds, columns=fractions) * 100
im = ax.imshow(H.values, aspect="auto", cmap="viridis")
ax.set_xticks(range(len(fractions)))
ax.set_xticklabels([f"{f:.0%}" for f in fractions])
ax.set_yticks(range(len(seeds)))
ax.set_yticklabels([f"Seed {s}" + (" ★" if s == best_seed else "") for s in seeds])
vmid = (np.nanmin(H.values) + np.nanmax(H.values)) / 2
for i in range(len(seeds)):
    for j in range(len(fractions)):
        v = H.values[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=9,
                color="white" if v < vmid else "black")
ax.set_xlabel("Labeled training data")
ax.set_title("Accuracy (%) — Seed × Fraction")
plt.colorbar(im, ax=ax, label="Accuracy (%)")

fig.suptitle("RoseSSL-CBAMNet — Label Efficiency: Seed-wise Comparison", fontsize=15, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.97])
out = os.path.join(OUTPUT_DIR, "label_efficiency_seed_comparison.png")
plt.savefig(out, dpi=600, bbox_inches="tight")
plt.show()
print("Saved:", out)